# Metadata-Aware Ensemble and Calibration Study

This notebook compares two **pre-specified** study configurations: `A0_BaseOnly` (no age/region metadata) and `A4_CHAMPION_G3_R_full` (age and anatomical-region metadata). It also compares five calibration pipelines. Within each study, the calibration method is selected using validation metrics; test metrics are computed only after those choices are frozen.

**Protocol guardrails**
- Ensemble weights are optimized using validation NLL only.
- Member-level and ensemble temperatures are selected using validation NLL; the final calibration method is selected on validation by NLL, with ECE as a tie-breaker.
- A0 and A4 are treated as pre-specified comparisons. The code preserves their configured order and does not select the metadata configuration based on test results.
- Test-set calibration comparisons, rescue diagnostics, and performance statistics are descriptive comparisons of pre-specified methods; do not use them to revise model configurations, weights, or calibration settings.
- The paired bootstrap confidence intervals resample lesion clusters. The McNemar p-value is computed at image level and should be interpreted with that limitation.
- The oracle metric is a non-deployable upper-bound diagnostic and is not used for model selection.

Before running, configure the data and feature paths using the environment variables listed in the configuration section. This repository should contain code and documentation, not the original datasets, derived feature arrays, or trained checkpoints.

## Research questions by code section

| Section | Research question | Selection/evaluation role |
|---|---|---|
| Environment and imports | Can experiments be run reproducibly with deterministic settings? | No model selection |
| Configuration | Which fixed experiment settings and two study protocols are compared? | Pre-specified A0 vs A4 |
| I/O helpers | Can split identifiers, labels, and previously extracted features be loaded consistently? | Input validation only |
| Metadata processing | How are age, anatomical region, and lesion identifiers mapped to image IDs? | Metadata preprocessing |
| Metadata augmentation | Does appending configured metadata features change the representation? | Used in A0/A4 feature construction |
| Classifier models | How are the MLP and logistic-regression members defined and trained? | Training on train split; tuning on validation where specified |
| Calibration and ensemble fusion | Which weights and temperatures minimize validation NLL? | Validation-only selection |
| Statistical tests | How are paired uncertainty intervals and McNemar comparisons computed? | Post-selection test comparison |
| Scaling/PCA | How are scaling and dimensionality reduction fitted without using test labels? | Train for validation; train+validation for final refit |
| Prespecified hyperparameters | What fixed hyperparameter presets are used? | Prespecified values / validation selection where explicitly implemented |
| Reporting and rescue helpers | How are confusion matrices, reports, and rescue diagnostics generated? | Final evaluation diagnostics |
| Core training and validation | How are member models and calibration parameters selected? | Train/validation only |
| Main experiment runner | How are frozen configurations evaluated and compared? | Test only after selection is complete |


## Environment and imports

Imports, runtime settings, and deterministic computation controls. Keep the CUBLAS setting before importing PyTorch.

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================
# B12-FINAL: A0 baseline vs. the pre-specified A4 metadata-enhanced configuration (G3_NoMiss + R_full)
# - Deterministic execution per study (reset RANDOM_STATE at the start of each study)
#   * Keeps A4 results reproducible regardless of whether A0 was run before it.
# - Feature processing: metadata augmentation (age/region) -> scaling -> PCA
# - Pipeline-consistent calibration comparison; selection uses validation NLL:
#   1) Uncalibrated (w_raw)
#   2) Conventional_TS (w_raw + post-ensemble T_conv)
#   3) Member_Only_TS (Stage-1 T_i + w_stage1)
#   4) TwoStage_Scalar_TS (Stage-1 T_i + w_stage1 + Stage-2 scalar T_ens)
#   5) Proposed_ClassAdaptive_TS (Stage-1 T_i + w_stage1 + Stage-2 L2-anchored T_1..T_K)
# - Rescue analysis, lesion-cluster paired bootstrap confidence intervals, and an image-level McNemar test
# ============================================================

# ---- This environment setting must be configured before importing torch.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json, warnings, copy, itertools
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, top_k_accuracy_score, log_loss,
    roc_auc_score, classification_report, confusion_matrix, cohen_kappa_score
)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from scipy.stats import chi2_contingency, binomtest, chi2 as chi2_dist
from scipy.optimize import minimize

import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import joblib

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## Configuration

Defines paths, random seed, PCA/calibration parameters, ensemble constraints, and output directory. These are configuration inputs, not learned from the test set.

In [ ]:
# -------------------- Configuration --------------------
# Override these defaults with environment variables in your own environment.
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = Path(os.environ.get("SPLIT_DIR", str(PROJECT_ROOT / "splits")))
FEATURE_DIRS = {
    "SWIN": Path(os.environ.get("SWIN_FEATURE_DIR", str(PROJECT_ROOT / "features" / "swin_ft768"))),
    "EVA": Path(os.environ.get("EVA_FEATURE_DIR", str(PROJECT_ROOT / "features" / "eva02"))),
    "COAT": Path(os.environ.get("COAT_FEATURE_DIR", str(PROJECT_ROOT / "features" / "coatnet_ft672"))),
}
META_CSV = Path(os.environ.get("META_CSV", str(PROJECT_ROOT / "metadata" / "ISIC_2019_Training_Metadata.csv")))
SAVE_DIR = Path(os.environ.get("SKIN_LESION_OUTPUT_DIR", str(PROJECT_ROOT / "outputs" / "ensemble_b12_final_seed_fixed")))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RANDOM_STATE = 42

USE_PCA = True
PCA_VARIANCE = 0.99
T_MIN, T_MAX, T_STEP = 0.70, 1.50, 0.02
MLP_BATCH = 256

# Ensemble weight search
COAT_FLOOR = 0.05
W_STEP = 0.02

# Pre-specified L2 prior anchoring for Class-Adaptive TS (tight anchoring around T_ens)
LAMBDA_CLASS_T = 0.50
CLASS_T_BOUND_DELTA = 0.04

# Number of bootstrap replicates
N_BOOTSTRAP = 1000

## Pre-specified Study Configurations

Defines A0 (without metadata) and A4 (with the pre-specified age and anatomical-region configuration). These configurations are compared as specified; the test scores must not be used to choose between them.

In [ ]:
# -------------------- Pre-specified Study Configurations --------------------
CHAMPION_AGE_CFG = {
    "edges": [(0,10),(10,20),(20,30),(30,40),(40,50),(50,60),(60,70),(70,80),(80,np.inf)],
    "add_missing": False
}

CHAMPION_REGION_SCHEME = "R_full"
REGION_SCHEMES = {
    CHAMPION_REGION_SCHEME: {"rare_merge": 0,  "top_k": None, "custom_map": None},
}

VALID_STUDIES = {}

# 1. A0 baseline: no metadata (age and anatomical region are excluded).
VALID_STUDIES["A0_BaseOnly"] = {
    "use_age": False, "age_scheme_cfg": None,
    "use_region": False, "region_scheme": None
}

# 2. A4 metadata-enhanced configuration: pre-specified age grouping + anatomical-region encoding.
VALID_STUDIES["A4_CHAMPION_G3_R_full"] = {
    "use_age": True, "age_scheme_cfg": CHAMPION_AGE_CFG,
    "use_region": True, "region_scheme": CHAMPION_REGION_SCHEME
}


def set_seed(seed=RANDOM_STATE):
    import random
    import torch.backends.cudnn as cudnn
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = False
    cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=False)

set_seed(RANDOM_STATE)
os.makedirs(SAVE_DIR, exist_ok=True)
print("Device:", DEVICE)
print(f"[INIT] Pre-specified study protocols: {list(VALID_STUDIES.keys())}")

## IO helpers

Loads split CSV files and pre-extracted feature arrays while normalizing image identifiers. No model selection occurs in this section.

In [ ]:
# -------------------- IO helpers --------------------
def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _read_split_csv(csv_path):
    df = pd.read_csv(csv_path)
    id_col = next((c for c in ["image_id","image","Image","filename","fname","img_id","name","file","img"] if c in df.columns), None)
    if id_col is None: raise ValueError(f"{csv_path}: No image_id/image column was found.")
    ids = _normalize_ids(df[id_col])
    if "label" in df.columns:
        y = df["label"].astype(int).values
        classes = sorted(pd.Series(y).unique().tolist())
        class_names = [f"class{i}" for i in classes]
    elif "class_name" in df.columns:
        class_names = sorted(df["class_name"].unique().tolist())
        cmap = {c:i for i,c in enumerate(class_names)}
        y = df["class_name"].map(cmap).astype(int).values
    else:
        raise ValueError(f"{csv_path}: Neither label nor class_name column was found.")
    return ids, y, class_names

def load_global_splits(split_dir):
    paths = {"train": os.path.join(split_dir, "train_split.csv"),
             "val"  : os.path.join(split_dir, "val_split.csv"),
             "test" : os.path.join(split_dir, "test_split.csv")}
    out = {}
    class_names = None
    for sp, pth in paths.items():
        ids, y, cls = _read_split_csv(pth)
        out[sp] = {"ids": ids, "y": y}
        if class_names is None: class_names = cls
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"Split features were not found in {dir_path} for split={split}.")
    return np.load(X_path)

## Metadata (Age, Region, and Lesion ID)

Reads and normalizes age, anatomical-region, and lesion identifiers. The lesion identifier is used for cluster-aware bootstrap resampling.

In [ ]:
# -------------------- Metadata (Age, Region, and Lesion ID) --------------------
AGE_ID_CANDS  = ["image_id","image","Image","filename","fname","name","file","img"]
AGE_COL_CANDS = ["age","Age","age_approx","patient_age","age_years","age_approximate"]
REGION_CANDS  = ["anatom_site_general","anatom_site","region","body_site","anatom_site_general_challenge"]
LESION_CANDS  = ["lesion_id","lesion","lesion_ID","Lesion_ID"]

def _read_meta(meta_csv):
    m = pd.read_csv(meta_csv)
    id_col = next((c for c in AGE_ID_CANDS if c in m.columns), None)
    if id_col is None: raise ValueError("The metadata CSV has no image ID column.")
    m["_img"] = _normalize_ids(m[id_col].astype(str).values)

    age_col = next((c for c in AGE_COL_CANDS if c in m.columns), None)
    if age_col is None:
        m["_age"] = np.nan
    else:
        m["_age"] = pd.to_numeric(m[age_col], errors="coerce").clip(lower=0)

    reg_col = next((c for c in REGION_CANDS if c in m.columns), None)
    if reg_col is None:
        m["_region_raw"] = "missing"
    else:
        reg = m[reg_col].astype(str).str.strip().str.lower().replace({"nan":"missing","none":"missing"})
        reg = reg.where(~reg.isin(["","nan","none","na"]), "missing")
        m["_region_raw"] = reg

    les_col = next((c for c in LESION_CANDS if c in m.columns), None)
    if les_col is None:
        m["_lesion_id"] = "SINGLETON_" + m["_img"]
    else:
        les = m[les_col].astype(str).str.strip()
        invalid_mask = les.str.lower().isin(["", "nan", "none", "na", "null"]) | m[les_col].isna()
        m["_lesion_id"] = np.where(invalid_mask, "SINGLETON_" + m["_img"], les)

    return m[["_img","_age","_region_raw","_lesion_id"]].drop_duplicates("_img", keep="first")

def get_lesion_clusters(ids, meta_df):
    les_mp = {r["_img"]: r["_lesion_id"] for _, r in meta_df.iterrows()}
    return np.array([les_mp.get(img, f"SINGLETON_{img}") for img in ids], dtype=object)

def _age_onehot(a, edges):
    v = np.zeros(len(edges), dtype=np.float32)
    if a is None or np.isnan(a): return v
    for i,(lo,hi) in enumerate(edges):
        if (a>=lo) and (a<hi): v[i]=1.0; break
    return v

def make_region_vocab_and_mapper(ids_train, meta_df, scheme_cfg):
    mp = {r["_img"]: r["_region_raw"] for _, r in meta_df.iterrows()}
    regs_tr = pd.Series([mp.get(i, "missing") for i in ids_train], dtype=str).fillna("missing").str.lower().str.strip()

    cm = scheme_cfg.get("custom_map", None)
    if cm is not None:
        regs_tr = regs_tr.replace(cm)

    vc = regs_tr.value_counts()
    top_k = scheme_cfg.get("top_k", None)
    rare_merge = scheme_cfg.get("rare_merge", 0)
    if top_k is not None:
        keep = vc.index[:top_k].tolist()
    elif rare_merge and rare_merge>0:
        keep = vc[vc>=rare_merge].index.tolist()
    else:
        keep = vc.index.tolist()

    vocab = sorted(pd.Series(keep + ["other","missing"]).unique().tolist())
    if "missing" in vocab:
        vocab = [v for v in vocab if v!="missing"] + ["missing"]
    if "other" in vocab and (len(vocab)==0 or vocab[-1]!="missing"):
        vocab = [v for v in vocab if v not in ["other","missing"]] + ["other"] + (["missing"] if "missing" in vocab else [])

    def map_ids(ids_any):
        regs = pd.Series([mp.get(i, "missing") for i in ids_any], dtype=str).fillna("missing").str.lower().str.strip()
        if cm is not None:
            regs = regs.replace(cm)
        return regs.where(regs.isin(keep), "other").to_dict()

    return vocab, map_ids

def build_region_ohe(ids, id2map, vocab):
    idx = {v:i for i,v in enumerate(vocab)}
    M = np.zeros((len(ids), len(vocab)), dtype=np.float32)
    miss_idx = idx.get("missing", None)
    for r, img in enumerate(ids):
        key = id2map.get(img, "missing")
        j = idx.get(key, miss_idx)
        if j is not None: M[r, j] = 1.0
    return M

def build_age_block(ids, meta_df, edges, add_missing=True):
    age_mp = {r["_img"]: r["_age"] for _, r in meta_df.iterrows()}
    rows=[]
    for img in ids:
        a = age_mp.get(img, np.nan)
        one = _age_onehot(a, edges)
        if add_missing:
            miss = 0.0 if (a is not None and not np.isnan(a)) else 1.0
            one = np.concatenate([one, [miss]]).astype(np.float32)
        rows.append(one.astype(np.float32))
    return np.stack(rows, 0)

## Metadata Feature Augmentation

Creates age-bin and anatomical-region feature blocks, then appends them to each image feature vector when enabled by the study configuration.

In [ ]:
# -------------------- Metadata Feature Augmentation --------------------
def augment(base_X, ids, use_age, use_region, meta_df, age_scheme_cfg=None, region_vocab=None, region_id2map=None):
    parts=[base_X]
    if use_age:
        assert age_scheme_cfg is not None, "use_age=True ama age_scheme_cfg=None!"
        parts.append(build_age_block(ids, meta_df,
                                   edges=age_scheme_cfg["edges"],
                                   add_missing=age_scheme_cfg["add_missing"]))
    if use_region:
        assert region_vocab is not None and region_id2map is not None
        parts.append(build_region_ohe(ids, region_id2map, region_vocab))

    if len(parts) == 1:
        return parts[0].astype(np.float32)

    return np.hstack(parts).astype(np.float32)

## Classifier Models

Defines the MLP and logistic-regression classifiers and their training/inference helpers.

In [ ]:
# -------------------- Classifier Models --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(2048,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

def train_mlp(Xfit, yfit, n_classes, hp, device, tag, cosine=False):
    hp = copy.deepcopy(hp); hp["batch_size"] = MLP_BATCH
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, hp["epochs"])) if cosine else None
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    loader = _loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        losses=[]
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
            losses.append(loss.item())
        if sched is not None: sched.step()
        if ep==1 or ep%max(1,hp.get("log_every",10))==0 or ep==hp["epochs"]:
            with torch.no_grad():
                P = torch.softmax(model(torch.tensor(Xfit, dtype=torch.float32, device=device)), dim=1).cpu().numpy()
            acc = accuracy_score(yfit, P.argmax(1))
            f1  = f1_score(yfit, P.argmax(1), average="macro", zero_division=0)
            print(f"[{tag}] ep={ep:03d} | loss={np.mean(losses):.4f} | acc={acc:.4f} | f1={f1:.4f}")
    return model

def logits_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); outs=[]
    with torch.no_grad():
        for xb,_ in loader:
            outs.append(model(xb.to(device)).cpu())
    lg = torch.cat(outs,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def make_logreg(hp):
    hp = hp.copy()
    if hp.get("solver","lbfgs") == "lbfgs": hp.pop("n_jobs", None)
    return LogisticRegression(**hp)

## Calibration and Ensemble Fusion

Implements temperature scaling and ensemble fusion. Weight searches use validation NLL; this section contains no test-based weight search.

In [ ]:
# -------------------- Calibration and Ensemble Fusion --------------------
def temp_scale(P, T):
    """Apply temperature scaling with scalar T or class-wise T_vec (shape: [K,])."""
    P = np.clip(P, 1e-12, 1.0)
    T_arr = np.asarray(T, dtype=np.float64)
    if T_arr.ndim == 1:
        T_arr = T_arr[None, :]
    log_Q = np.log(P) / T_arr
    log_Q = log_Q - np.max(log_Q, axis=1, keepdims=True)
    Q = np.exp(log_Q)
    Q = Q / Q.sum(axis=1, keepdims=True)
    return Q

def pick_T_by_nll(y_true, P, n_class, tmin, tmax, tstep):
    grid = np.arange(tmin, tmax+1e-9, tstep)
    bestT, bestNLL = 1.0, 1e18
    for T in grid:
        Q = temp_scale(P, T)
        nll = log_loss(y_true, np.clip(Q, 1e-12, 1.0), labels=list(range(n_class)))
        if nll < bestNLL: bestNLL, bestT = nll, T
    return float(bestT), float(bestNLL)

def fit_class_adaptive_T(y_val, P_val_stage1, T_init, n_class, lambda_reg=LAMBDA_CLASS_T, bound_delta=CLASS_T_BOUND_DELTA):
    """
    Stage-2 NLL + L2-Regularized Class-Adaptive Temperature Scaling:
        L(T) = NLL(temp_scale(P_val_stage1, T)) + lambda_reg * mean((T - T_init)^2)
    Optimize only NLL plus L2 regularization, using the pre-specified tight L2 anchor (lambda_reg=0.50, bound_delta=0.04).
    """
    T_vec = np.full(n_class, float(T_init), dtype=np.float64)
    labels_list = list(range(n_class))

    def _objective(t_arr):
        Q = temp_scale(P_val_stage1, t_arr)
        Q_c = np.clip(Q, 1e-12, 1.0)
        nll = float(log_loss(y_val, Q_c, labels=labels_list))
        reg = float(lambda_reg * np.mean((t_arr - T_init) ** 2))
        return nll + reg

    best_obj = _objective(T_vec)
    bounds = [(max(0.65, T_init - bound_delta), min(1.45, T_init + bound_delta)) for _ in range(n_class)]

    res = minimize(_objective, T_vec, method="L-BFGS-B", bounds=bounds, options={"maxiter": 100, "ftol": 1e-9})
    if res.success and res.fun < best_obj:
        T_vec = res.x.copy()
        best_obj = float(res.fun)

    for _pass in range(2):
        improved = False
        for k in range(n_class):
            grid_k = np.linspace(bounds[k][0], bounds[k][1], 41)
            for tk in grid_k:
                T_try = T_vec.copy()
                T_try[k] = tk
                obj_try = _objective(T_try)
                if obj_try < best_obj - 1e-7:
                    best_obj = obj_try
                    T_vec = T_try
                    improved = True
        if not improved:
            break
    return T_vec

def compute_ece(y_true, P, n_bins=15):
    confidences = np.max(P, axis=1)
    predictions = np.argmax(P, axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower, bin_upper = bin_boundaries[i], bin_boundaries[i + 1]
        in_bin = (confidences >= bin_lower) & (confidences <= bin_upper) if i == 0 else (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            ece += np.abs(np.mean(confidences[in_bin]) - np.mean(accuracies[in_bin])) * prop_in_bin
    return float(ece)

def mix3(Pa, Pb, Pc, w_a, w_b):
    w_c = 1.0 - w_a - w_b
    return w_a*Pa + w_b*Pb + w_c*Pc

def weight_search_nll(P_val_dict, y_val, n_class):
    """Search ensemble fusion weights using a proper scoring rule (minimize validation NLL)."""
    best = {"w_swin":0.54, "w_eva":0.32, "w_coat":0.14, "score":1e18}
    grid = np.arange(0.0, 1.0+1e-9, W_STEP)
    labels_list = list(range(n_class))
    for ws in grid:
        for we in grid:
            wc = 1.0 - ws - we
            if wc < COAT_FLOOR - 1e-12: continue
            w = np.array([max(0.0,ws), max(0.0,we), max(COAT_FLOOR,wc)], dtype=float); w/=w.sum()
            Pmix = mix3(P_val_dict["SWIN"], P_val_dict["EVA"], P_val_dict["COAT"], w[0], w[1])
            score = log_loss(y_val, np.clip(Pmix,1e-12,1.0), labels=labels_list)
            if score < best["score"] - 1e-12:
                best = {"w_swin":float(w[0]), "w_eva":float(w[1]), "w_coat":float(w[2]), "score":float(score)}
    return best

def eval_calib_metrics(y_true, P, n_class):
    P_c = np.clip(P, 1e-12, 1.0)
    P_c = P_c / P_c.sum(axis=1, keepdims=True)
    y_pred = P_c.argmax(1)
    Y_ohe = np.eye(n_class)[y_true]

    ece   = compute_ece(y_true, P_c, n_bins=15)
    nll   = float(log_loss(y_true, P_c, labels=list(range(n_class))))
    brier = float(np.mean(np.sum((P_c - Y_ohe) ** 2, axis=1)))
    acc   = float(accuracy_score(y_true, y_pred))
    bacc  = float(balanced_accuracy_score(y_true, y_pred))
    mf1   = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    top3  = float(top_k_accuracy_score(y_true, P_c, k=3))
    try:
        Y_bin = label_binarize(y_true, classes=list(range(n_class)))
        auc = float(roc_auc_score(Y_bin, P_c, average="macro", multi_class="ovr"))
    except Exception:
        auc = np.nan
    return {"ECE": ece, "NLL": nll, "Brier": brier, "Acc": acc, "Balanced_Acc": bacc, "MacroF1": mf1, "Top3": top3, "ROC_AUC": auc}

## Statistical Tests (Lesion-Level Cluster Bootstrap)

Defines paired bootstrap confidence intervals and an image-level McNemar test. The bootstrap calls should pass test lesion IDs to preserve lesion clusters.

In [ ]:
# -------------------- Statistical Tests (Lesion-Level Cluster Bootstrap) --------------------
def mcnemar_test(y_true, pred_a, pred_b):
    ok_a = (pred_a == y_true)
    ok_b = (pred_b == y_true)
    n01 = int(np.sum((~ok_a) & ok_b))
    n10 = int(np.sum(ok_a & (~ok_b)))
    n_disc = n01 + n10
    if n_disc == 0:
        return {"n_B_better": 0, "n_A_better": 0, "chi2_cc": 0.0, "p_exact": 1.0, "p_chi2": 1.0}
    chi2_cc = ((abs(n01 - n10) - 1.0) ** 2) / n_disc
    p_chi2 = float(1.0 - chi2_dist.cdf(chi2_cc, df=1))
    p_exact = float(binomtest(min(n01, n10), n=n_disc, p=0.5, alternative="two-sided").pvalue)
    return {"n_B_better": n01, "n_A_better": n10, "chi2_cc": float(chi2_cc), "p_exact": p_exact, "p_chi2": p_chi2}

def paired_bootstrap_ci(y_true, probs_dict, ref_method, target_method, n_class, lesion_ids=None, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    N = len(y_true)
    P_ref = np.clip(probs_dict[ref_method], 1e-12, 1.0)
    P_ref = P_ref / P_ref.sum(axis=1, keepdims=True)
    P_tar = np.clip(probs_dict[target_method], 1e-12, 1.0)
    P_tar = P_tar / P_tar.sum(axis=1, keepdims=True)

    Y_ohe = np.eye(n_class)[y_true]
    pred_ref = P_ref.argmax(1)
    pred_tar = P_tar.argmax(1)

    if lesion_ids is not None:
        unique_clusters = pd.unique(lesion_ids)
        cluster_to_indices = {}
        for idx_i, lid in enumerate(lesion_ids):
            cluster_to_indices.setdefault(lid, []).append(idx_i)
        cluster_idx_arrays = [np.array(cluster_to_indices[lid], dtype=int) for lid in unique_clusters]
        n_clusters = len(cluster_idx_arrays)
        sampling_mode = "LesionCluster"
    else:
        cluster_idx_arrays = None
        n_clusters = N
        sampling_mode = "ImageLevel"

    metrics = ["Acc", "MacroF1", "ECE", "NLL", "Brier"]
    diffs = {m: np.zeros(n_boot, dtype=np.float64) for m in metrics}
    tar_vals = {m: np.zeros(n_boot, dtype=np.float64) for m in metrics}

    for b in range(n_boot):
        if cluster_idx_arrays is not None:
            sampled_c = rng.integers(0, n_clusters, size=n_clusters)
            idx = np.concatenate([cluster_idx_arrays[c] for c in sampled_c])
        else:
            idx = rng.integers(0, N, size=N)

        yb = y_true[idx]
        Y_ohe_b = Y_ohe[idx]

        pt = P_tar[idx]
        prt = pred_tar[idx]
        acc_t = np.mean(prt == yb)
        mf1_t = f1_score(yb, prt, average="macro", zero_division=0)
        ece_t = compute_ece(yb, pt, n_bins=15)
        nll_t = -np.mean(np.sum(Y_ohe_b * np.log(pt), axis=1))
        br_t  = np.mean(np.sum((pt - Y_ohe_b) ** 2, axis=1))

        pr = P_ref[idx]
        prr = pred_ref[idx]
        acc_r = np.mean(prr == yb)
        mf1_r = f1_score(yb, prr, average="macro", zero_division=0)
        ece_r = compute_ece(yb, pr, n_bins=15)
        nll_r = -np.mean(np.sum(Y_ohe_b * np.log(pr), axis=1))
        br_r  = np.mean(np.sum((pr - Y_ohe_b) ** 2, axis=1))

        tar_vals["Acc"][b], tar_vals["MacroF1"][b], tar_vals["ECE"][b], tar_vals["NLL"][b], tar_vals["Brier"][b] = acc_t, mf1_t, ece_t, nll_t, br_t
        diffs["Acc"][b]     = acc_t - acc_r
        diffs["MacroF1"][b] = mf1_t - mf1_r
        diffs["ECE"][b]     = ece_t - ece_r
        diffs["NLL"][b]     = nll_t - nll_r
        diffs["Brier"][b]   = br_t - br_r

    mcn = mcnemar_test(y_true, pred_ref, pred_tar)
    rows = []
    for m in metrics:
        d = diffs[m]
        tv = tar_vals[m]
        rows.append({
            "Comparison": f"{target_method} vs {ref_method}",
            "Bootstrap_Type": sampling_mode,
            "N_Clusters": int(n_clusters),
            "Metric": m,
            "Target_Mean": float(np.mean(tv)),
            "Target_CI95_Low": float(np.percentile(tv, 2.5)),
            "Target_CI95_High": float(np.percentile(tv, 97.5)),
            "Delta_Mean": float(np.mean(d)),
            "Delta_CI95_Low": float(np.percentile(d, 2.5)),
            "Delta_CI95_High": float(np.percentile(d, 97.5)),
            "McNemar_p_exact": mcn["p_exact"] if m == "Acc" else np.nan,
            "McNemar_n_TargetBetter": mcn["n_B_better"] if m == "Acc" else np.nan,
            "McNemar_n_RefBetter": mcn["n_A_better"] if m == "Acc" else np.nan
        })
    return rows

def compute_error_diversity(y_true, P_members_dict):
    names = ["SWIN", "EVA", "COAT"]
    preds = {k: P_members_dict[k].argmax(1) for k in names}
    errs  = {k: (preds[k] != y_true).astype(int) for k in names}

    rows = []
    for a, b in itertools.combinations(names, 2):
        pa, pb = preds[a], preds[b]
        ea, eb = errs[a], errs[b]
        disagreement = float(np.mean(pa != pb))
        kappa = float(cohen_kappa_score(pa, pb))
        double_fault = float(np.mean((ea == 1) & (eb == 1)))
        err_corr = float(np.corrcoef(ea, eb)[0, 1]) if (np.std(ea) > 0 and np.std(eb) > 0) else 0.0
        only_a_correct = int(np.sum((ea == 0) & (eb == 1)))
        only_b_correct = int(np.sum((ea == 1) & (eb == 0)))
        rows.append({
            "Pair": f"{a} vs {b}",
            "Disagreement": disagreement,
            "Cohens_Kappa": kappa,
            "Double_Fault": double_fault,
            "Error_Correlation": err_corr,
            "Only_First_Correct": only_a_correct,
            "Only_Second_Correct": only_b_correct
        })
    return pd.DataFrame(rows)

## Scale/PCA

Fits scalers/PCA on training data during validation selection and refits the transform on train+validation for final test evaluation.

In [ ]:
# -------------------- Scale/PCA --------------------
def fit_transform_train_then_val(Xtr_aug, Xva_aug, use_pca=True, pca_var=0.99):
    sc = StandardScaler().fit(Xtr_aug)
    Ztr, Zva = sc.transform(Xtr_aug), sc.transform(Xva_aug)
    if not use_pca: return sc, None, Ztr, Zva, Ztr.shape[1]

    n_components = min(Xtr_aug.shape[0], Xtr_aug.shape[1])
    if isinstance(pca_var, float) and pca_var < 1.0:
        pca_n_arg = min(n_components, int(Xtr_aug.shape[0] * 0.99) - 1)
        if pca_n_arg < 0.9 * min(Xtr_aug.shape):
             solver = "randomized"
        else:
             solver = "full"

        pca_temp = PCA(n_components=max(2, pca_n_arg), svd_solver=solver, random_state=RANDOM_STATE).fit(Ztr)
        cumsum = np.cumsum(pca_temp.explained_variance_ratio_)
        k = np.argmax(cumsum >= pca_var) + 1
        pca = PCA(n_components=int(k), svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    else:
        k = int(min(pca_var, n_components))
        pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)

    Ztr_p, Zva_p = pca.transform(Ztr), pca.transform(Zva)
    return sc, pca, Ztr_p, Zva_p, Ztr_p.shape[1]


def fit_transform_trainval_then_test_fixed(Xtr_aug, Xva_aug, Xte_aug, use_pca=True, pca_n=None, pca_var=0.99):
    Xtrval = np.vstack([Xtr_aug, Xva_aug])
    sc = StandardScaler().fit(Xtrval)
    Ztr, Zva, Zte = sc.transform(Xtr_aug), sc.transform(Xva_aug), sc.transform(Xte_aug)
    if not use_pca: return sc, None, Ztr, Zva, Zte

    Ztrval = np.vstack([Ztr, Zva])
    n_samples, n_features = Ztrval.shape

    if pca_n is None:
        n_components = min(n_samples, n_features)
        pca_n_arg = min(n_components, int(n_samples * 0.99) - 1)
        if pca_n_arg < 0.9 * min(n_samples, n_features):
             solver = "randomized"
        else:
             solver = "full"
        pca_temp = PCA(n_components=max(2, pca_n_arg), svd_solver=solver, random_state=RANDOM_STATE).fit(Ztrval)
        cumsum = np.cumsum(pca_temp.explained_variance_ratio_)
        k = np.argmax(cumsum >= pca_var) + 1
        pca = PCA(n_components=int(k), svd_solver="full", random_state=RANDOM_STATE).fit(Ztrval)
    else:
        k = int(min(pca_n, n_samples, n_features))
        pca = PCA(n_components=k, svd_solver="full", random_state=RANDOM_STATE).fit(Ztrval)

    return sc, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

## Prespecified Hyperparameters

Stores the configured MLP and logistic-regression hyperparameters used by the experiment.

In [ ]:
# -------------------- Prespecified Hyperparameters --------------------
NAMED_PRESETS = {
    "SwinMLP":{"hidden":[2048],"dropout":0.1605757337,"lr":3.41409509895e-4,"weight_decay":1.488348195284e-4,"epochs":70,"log_every":10},
    "EvaMLP": {"hidden":[2048],"dropout":0.3203382207442766,"lr":9.765455593461e-4,"weight_decay":8.212522451000109e-05,"epochs":70,"log_every":10},
}
HP_LOGREG = {"C":2.3402196063650043,"max_iter":587,"tol":3.6e-04,"solver":"lbfgs","class_weight":None,"n_jobs":-1,"multi_class":"auto","random_state":RANDOM_STATE}

## Confusion Matrix, Classification Report, and Rescue Helpers

Writes final reporting artifacts and computes the rescue-attribution diagnostic for the selected pipeline.

In [ ]:
# -------------------- Confusion Matrix, Classification Report, and Rescue Helpers --------------------
def save_cm_cr(out_dir, y_true, y_pred, class_names, study_key):
    os.makedirs(out_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm,
                         index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cm_path = os.path.join(out_dir, f"CM_{study_key}.csv")
    cm_df.to_csv(cm_path, index=True)
    cr_txt = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
    with open(os.path.join(out_dir, f"CR_{study_key}.txt"), "w", encoding="utf-8") as f:
        f.write(cr_txt)
    cr_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(cr_dict).to_csv(os.path.join(out_dir, f"CR_{study_key}.csv"))

def leave_one_out_rescue(P_s, P_e, P_c, weights, y_true, T_stage2=None):
    ws, we, wc = weights
    P_full = ws*P_s + we*P_e + wc*P_c

    denom = max(we + wc, 1e-12)
    P_minus_s = (we/denom)*P_e + (wc/denom)*P_c

    denom = max(ws + wc, 1e-12)
    P_minus_e = (ws/denom)*P_s + (wc/denom)*P_c

    denom = max(ws + we, 1e-12)
    P_minus_c = (ws/denom)*P_s + (we/denom)*P_e

    if T_stage2 is not None:
        P_full    = temp_scale(P_full, T_stage2)
        P_minus_s = temp_scale(P_minus_s, T_stage2)
        P_minus_e = temp_scale(P_minus_e, T_stage2)
        P_minus_c = temp_scale(P_minus_c, T_stage2)

    y_full    = P_full.argmax(1)
    y_minus_s = P_minus_s.argmax(1)
    y_minus_e = P_minus_e.argmax(1)
    y_minus_c = P_minus_c.argmax(1)

    ok_full = (y_full == y_true)
    rescue_s = (ok_full) & (y_minus_s != y_true)
    rescue_e = (ok_full) & (y_minus_e != y_true)
    rescue_c = (ok_full) & (y_minus_c != y_true)

    N = len(y_true)
    rows = []
    for i in range(N):
        rescued_by = []
        if rescue_s[i]: rescued_by.append("SWIN")
        if rescue_e[i]: rescued_by.append("EVA")
        if rescue_c[i]: rescued_by.append("COAT")
        rows.append({
            "idx": i,
            "y_true": int(y_true[i]),
            "ens_pred": int(y_full[i]),
            "minus_s_pred": int(y_minus_s[i]),
            "minus_e_pred": int(y_minus_e[i]),
            "minus_c_pred": int(y_minus_c[i]),
            "rescued_by": "|".join(rescued_by) if rescued_by else ""
        })

    df = pd.DataFrame(rows)
    counts = {
        "SWIN": int(rescue_s.sum()),
        "EVA" : int(rescue_e.sum()),
        "COAT": int(rescue_c.sum())
    }
    return df, counts

def save_rescue_reports(out_dir, ids_te, class_names, y_true, P_test_dict, weights, T_stage2=None):
    df, counts = leave_one_out_rescue(
        P_test_dict["SWIN"], P_test_dict["EVA"], P_test_dict["COAT"],
        weights, y_true, T_stage2=T_stage2
    )
    df["image_id"] = ids_te
    df = df[["idx","image_id","y_true","ens_pred","minus_s_pred","minus_e_pred","minus_c_pred","rescued_by"]]
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, "RESCUE_ATTRIBUTION.csv"), index=False)
    pd.DataFrame([counts]).to_csv(os.path.join(out_dir, "RESCUE_COUNTS.csv"), index=False)
    print(f"[RESCUE (Stage-2 Aligned)] counts -> SWIN={counts['SWIN']} | EVA={counts['EVA']} | COAT={counts['COAT']}")

def write_compare_cm_cr(save_dir, tag, class_names, A_ytrue, A_ypred, B_ytrue, B_ypred, A_name="A0_BaseOnly", B_name="BEST_META"):
    out_dir = os.path.join(save_dir, f"COMPARE_CM_CR_{tag}")
    os.makedirs(out_dir, exist_ok=True)
    save_cm_cr(out_dir, A_ytrue, A_ypred, class_names, f"{A_name}")
    save_cm_cr(out_dir, B_ytrue, B_ypred, class_names, f"{B_name}")
    summ = []
    for name, yt, yp in [(A_name, A_ytrue, A_ypred), (B_name, B_ytrue, B_ypred)]:
        acc = accuracy_score(yt, yp)
        mf1 = f1_score(yt, yp, average="macro", zero_division=0)
        summ.append({"Study": name, "Acc": acc, "MacroF1": mf1})
    pd.DataFrame(summ).to_csv(os.path.join(out_dir, "SUMMARY_A0_vs_A4.csv"), index=False)

## Core Training and Validation

Trains members, searches the CoAtNet logistic-regression C value using validation NLL, optimizes ensemble weights on validation NLL, and selects the calibration method on validation.

In [ ]:
# -------------------- Core Training and Validation --------------------
def train_val_members(data, class_names, splits, meta_df, use_age, age_scheme_cfg, use_region, region_vocab=None, idmap_tr=None, idmap_va=None):
    n_class = len(class_names)
    P_val, T_sel, aux = {}, {}, {}
    P_val_uncal = {}

    for memb in ["SWIN","EVA","COAT"]:
        Xtrb, Xvab = data[memb]["Xtr"], data[memb]["Xva"]
        ids_tr, ids_va = splits["train"]["ids"], splits["val"]["ids"]
        Xtr_aug = augment(Xtrb, ids_tr, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_tr if use_region else None)
        Xva_aug = augment(Xvab, ids_va, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_va if use_region else None)

        if Xtr_aug.shape[1] == 0:
            raise ValueError(f"Xtr_aug has zero features for {memb}! age_cfg={age_scheme_cfg}")

        sc, pca, Ztr, Zva, k = fit_transform_train_then_val(Xtr_aug, Xva_aug, USE_PCA, PCA_VARIANCE)

        if memb in ["SWIN","EVA"]:
            hp = copy.deepcopy(NAMED_PRESETS["SwinMLP" if memb=="SWIN" else "EvaMLP"])
            mdl = train_mlp(Ztr, data[memb]["ytr"], n_class, hp, DEVICE, f"{memb} TRAIN (VAL){'+META' if (use_age or use_region) else ''}")
            _, P = logits_proba(mdl, Zva, DEVICE)

            P_val_uncal[memb] = P.copy()
            T_i, _ = pick_T_by_nll(data[memb]["yva"], P, n_class, T_MIN, T_MAX, T_STEP)
            P_val[memb] = temp_scale(P, T_i)
            T_sel[memb] = float(T_i)
            aux[memb] = {"hp": hp, "pca_n": k}
        else:
            hpC = copy.deepcopy(HP_LOGREG); hpC.update({"class_weight":"balanced"})
            logreg = make_logreg(hpC); logreg.fit(Ztr, data["COAT"]["ytr"])
            P_raw = logreg.predict_proba(Zva)

            P_val_uncal[memb] = P_raw.copy()
            Tc, _ = pick_T_by_nll(data["COAT"]["yva"], P_raw, n_class, T_MIN, T_MAX, T_STEP)
            P_val[memb] = temp_scale(P_raw, Tc)
            T_sel[memb] = float(Tc)
            aux[memb] = {"hp":hpC, "pca_n":k}

    aux["_calib_extra"] = {"P_val_uncal": P_val_uncal}
    return P_val, T_sel, aux

def coat_val_grid_nll(data, splits, meta_df, use_age, age_scheme_cfg, use_region, region_vocab, idmap_tr, idmap_va, n_class):
    baseC = HP_LOGREG["C"]
    C_grid = [1.6, baseC, 3.2]
    bestC, bestNLL, bestP, bestT, bestP_raw = None, 1e18, None, None, None
    y_val = data["COAT"]["yva"]
    Xtrb, Xvab = data["COAT"]["Xtr"], data["COAT"]["Xva"]
    ids_tr, ids_va = splits["train"]["ids"], splits["val"]["ids"]
    for C in C_grid:
        Xtr_aug = augment(Xtrb, ids_tr, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_tr if use_region else None)
        Xva_aug = augment(Xvab, ids_va, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_va if use_region else None)
        sc, pca, Ztr, Zva, _ = fit_transform_train_then_val(Xtr_aug, Xva_aug, USE_PCA, PCA_VARIANCE)
        hpC = copy.deepcopy(HP_LOGREG); hpC.update({"class_weight":"balanced", "C":C})
        lg = make_logreg(hpC).fit(Ztr, data["COAT"]["ytr"])
        P_raw = lg.predict_proba(Zva)
        Tc,_ = pick_T_by_nll(y_val, P_raw, n_class, T_MIN, T_MAX, T_STEP)
        P = temp_scale(P_raw, Tc)
        nll = log_loss(y_val, np.clip(P,1e-12,1.0), labels=list(range(n_class)))
        if nll < bestNLL: bestNLL, bestC, bestP, bestT, bestP_raw = nll, C, P, Tc, P_raw
    return bestC, bestP, bestT, bestP_raw

def refit_and_test(data, aux, T_sel, splits, meta_df, use_age, age_scheme_cfg, use_region, region_vocab, idmap_tr, idmap_va, idmap_te, n_class):
    P_test = {}
    P_test_uncal = {}

    for memb in ["SWIN","EVA","COAT"]:
        Xtrb, Xvab, Xteb = data[memb]["Xtr"], data[memb]["Xva"], data[memb]["Xte"]
        ids_tr, ids_va, ids_te = splits["train"]["ids"], splits["val"]["ids"], splits["test"]["ids"]
        Xtr_aug = augment(Xtrb, ids_tr, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_tr if use_region else None)
        Xva_aug = augment(Xvab, ids_va, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_va if use_region else None)
        Xte_aug = augment(Xteb, ids_te, use_age, use_region, meta_df, age_scheme_cfg, region_vocab, idmap_te if use_region else None)

        sc, pca, Ztr, Zva, Zte = fit_transform_trainval_then_test_fixed(
            Xtr_aug, Xva_aug, Xte_aug, USE_PCA, aux[memb]["pca_n"], PCA_VARIANCE
        )

        if memb in ["SWIN","EVA"]:
            hp = aux[memb]["hp"]
            mdl = train_mlp(np.vstack([Ztr,Zva]), np.concatenate([data[memb]["ytr"], data[memb]["yva"]]),
                            n_class, hp, DEVICE, f"{memb} TRAIN∪VAL (TEST){'+META' if (use_age or use_region) else ''}")
            _, P = logits_proba(mdl, Zte, DEVICE)
            P_test_uncal[memb] = P.copy()
            P_test[memb]       = temp_scale(P, T_sel[memb])
        else:
            hpC = copy.deepcopy(aux[memb]["hp"])
            lg = make_logreg(hpC).fit(np.vstack([Ztr,Zva]), np.concatenate([data[memb]["ytr"], data[memb]["yva"]]))
            P_raw = lg.predict_proba(Zte)
            P_test_uncal[memb] = P_raw.copy()
            P_test[memb]       = temp_scale(P_raw, T_sel[memb])

    return P_test, P_test_uncal

def oracle_acc_top3(y_true, members_P):
    preds = [P.argmax(1) for P in members_P]
    oracle_acc = np.mean([any(p[i]==y_true[i] for p in preds) for i in range(len(y_true))])
    def topk(arr,k): return np.argpartition(-arr, k-1, axis=1)[:, :k]
    top3s = [topk(P,3) for P in members_P]
    oracle_top3 = np.mean([y_true[i] in np.unique(np.concatenate([t[i] for t in top3s])) for i in range(len(y_true))])
    return oracle_acc, oracle_top3

## Main experiment: validation selection and frozen test evaluation

Runs the pre-specified A0/A4 studies, evaluates their frozen configurations, and compares A0 directly against the pre-specified A4 configuration without ranking configurations by test scores.

**Research question:** Does the pre-specified metadata-enhanced configuration differ from the no-metadata baseline when all model and calibration decisions are made before test evaluation?

In [ ]:
# -------------------- Main Experiment Runner --------------------
def run_one_study(study_key, data, class_names, splits, meta_df):
    # IMPORTANT: Reset the RNG seed at the start of each study so that running A0 does not affect A4.
    set_seed(RANDOM_STATE)

    cfg = VALID_STUDIES[study_key]
    use_age, use_region = cfg["use_age"], cfg["use_region"]
    age_cfg = cfg["age_scheme_cfg"]
    rscheme = cfg["region_scheme"]

    print("\n" + "="*72)
    age_desc = f"G_key={age_cfg['edges'] if (age_cfg and 'edges' in age_cfg) else 'N/A'}" if use_age else "False"
    print(f"RUN -> {study_key} | age={age_desc} | region={use_region} | scheme={rscheme if rscheme else 'none'}")
    print("="*72)

    region_vocab = None; map_ids = None
    idmap_tr = idmap_va = idmap_te = None
    if use_region:
        region_vocab, map_ids = make_region_vocab_and_mapper(splits["train"]["ids"], meta_df, REGION_SCHEMES[rscheme])
        idmap_tr = map_ids(splits["train"]["ids"])
        idmap_va = map_ids(splits["val"]["ids"])
        idmap_te = map_ids(splits["test"]["ids"])

    # Validation members (Stage 1: estimate member-level T_i using validation NLL).
    P_val, T_sel, aux = train_val_members(data, class_names, splits, meta_df, use_age, age_cfg, use_region, region_vocab, idmap_tr, idmap_va)
    n_class = len(class_names)
    y_val = data["SWIN"]["yva"]

    # COAT C-grid (VAL)
    bestC, P_val_coat_best, T_coat_best, P_val_coat_raw = coat_val_grid_nll(
        data, splits, meta_df, use_age, age_cfg, use_region, region_vocab, idmap_tr, idmap_va, n_class
    )
    P_val["COAT"] = P_val_coat_best; T_sel["COAT"] = float(T_coat_best)
    aux["COAT"]["hp"]["class_weight"]="balanced"; aux["COAT"]["hp"]["C"]=float(bestC)

    P_val_uncal = aux["_calib_extra"]["P_val_uncal"]
    P_val_uncal["COAT"] = P_val_coat_raw

    # --- PIPELINE-CONSISTENT FUSION WEIGHTS (validation NLL optimum) ---
    w_raw    = weight_search_nll(P_val_uncal, y_val, n_class)
    w_stage1 = weight_search_nll(P_val,       y_val, n_class)

    print(f"[WEIGHTS w_raw (NLL-Optimal on Raw VAL)] "
          f"SWIN={w_raw['w_swin']:.2f} | EVA={w_raw['w_eva']:.2f} | COAT={w_raw['w_coat']:.2f} (val_NLL={w_raw['score']:.5f})")
    print(f"[WEIGHTS w_stage1 (NLL-Optimal on Stage-1 VAL)] "
          f"SWIN={w_stage1['w_swin']:.2f} | EVA={w_stage1['w_eva']:.2f} | COAT={w_stage1['w_coat']:.2f} (val_NLL={w_stage1['score']:.5f})")

    # --- VALIDATION ENSEMBLE PROBABILITIES AND TEMPERATURE FITTING ---
    # A. Raw-Space Family (uses w_raw)
    P_ens_val_raw = mix3(P_val_uncal["SWIN"], P_val_uncal["EVA"], P_val_uncal["COAT"], w_raw["w_swin"], w_raw["w_eva"])
    T_conv, _ = pick_T_by_nll(y_val, P_ens_val_raw, n_class, T_MIN, T_MAX, 0.005)
    P_ens_val_conv = temp_scale(P_ens_val_raw, T_conv)

    # B. Stage-1 Calibrated Family (uses w_stage1)
    P_ens_val_stage1 = mix3(P_val["SWIN"], P_val["EVA"], P_val["COAT"], w_stage1["w_swin"], w_stage1["w_eva"])
    T_ens, _ = pick_T_by_nll(y_val, P_ens_val_stage1, n_class, T_MIN, T_MAX, 0.005)
    P_ens_val_twostage_scalar = temp_scale(P_ens_val_stage1, T_ens)

    T_vec_class = fit_class_adaptive_T(
        y_val, P_ens_val_stage1, T_ens, n_class,
        lambda_reg=LAMBDA_CLASS_T, bound_delta=CLASS_T_BOUND_DELTA
    )
    P_ens_val_class_adaptive = temp_scale(P_ens_val_stage1, T_vec_class)

    print(f"[CALIB PARAMS ON VAL] T_conv(post-ens)={T_conv:.4f} | T_members={T_sel} | T_ens(scalar)={T_ens:.4f}")
    print(f"[CLASS-ADAPTIVE T_k ON VAL (lambda={LAMBDA_CLASS_T}, bound=+-{CLASS_T_BOUND_DELTA})] "
          f"{[round(float(x), 4) for x in T_vec_class]}")

    method_meta = {
        "Uncalibrated":              {"w_src": "w_raw(NLL)",    "w": w_raw},
        "Conventional_TS":           {"w_src": "w_raw(NLL)",    "w": w_raw},
        "Member_Only_TS":            {"w_src": "w_stage1(NLL)", "w": w_stage1},
        "TwoStage_Scalar_TS":        {"w_src": "w_stage1(NLL)", "w": w_stage1},
        "Proposed_ClassAdaptive_TS": {"w_src": "w_stage1(NLL)", "w": w_stage1},
    }

    # --- 1. VALIDATION EVALUATION ---
    val_ens_probs = {
        "Uncalibrated": P_ens_val_raw,
        "Conventional_TS": P_ens_val_conv,
        "Member_Only_TS": P_ens_val_stage1,
        "TwoStage_Scalar_TS": P_ens_val_twostage_scalar,
        "Proposed_ClassAdaptive_TS": P_ens_val_class_adaptive
    }
    val_ablation_rows = []
    for m_name, P_ens_v in val_ens_probs.items():
        mv = eval_calib_metrics(y_val, P_ens_v, n_class)
        w_used = method_meta[m_name]["w"]
        w_str = f"[{w_used['w_swin']:.2f}, {w_used['w_eva']:.2f}, {w_used['w_coat']:.2f}]"
        val_ablation_rows.append({
            "Study": study_key, "Split": "VAL", "Method": m_name,
            "Weight_Source": method_meta[m_name]["w_src"], "Weights": w_str, **mv
        })

    val_ablation_df = pd.DataFrame(val_ablation_rows)
    selected_calib_method = val_ablation_df.sort_values(["NLL", "ECE"], ascending=[True, True]).iloc[0]["Method"]
    val_ablation_df["Selected_on_VAL"] = (val_ablation_df["Method"] == selected_calib_method)

    print("\n--- [VALIDATION] 5-WAY PIPELINE-CONSISTENT CALIBRATION COMPARISON ---")
    print(val_ablation_df[["Method", "Weight_Source", "Weights", "ECE", "NLL", "Brier", "Acc", "Balanced_Acc", "MacroF1", "Selected_on_VAL"]].to_string(index=False))
    print(f"[VAL SELECTION] Selected method on validation set: {selected_calib_method}")

    # TEST refit & evaluate
    P_test, P_test_uncal = refit_and_test(
        data, aux, T_sel, splits, meta_df, use_age, age_cfg, use_region, region_vocab, idmap_tr, idmap_va, idmap_te, n_class
    )
    y_te = data["SWIN"]["yte"]

    # --- TEST ENSEMBLE PROBABILITIES (apply the frozen validation-selected parameters and weights) ---
    P_ens_test_raw             = mix3(P_test_uncal["SWIN"], P_test_uncal["EVA"], P_test_uncal["COAT"], w_raw["w_swin"], w_raw["w_eva"])
    P_ens_test_conv            = temp_scale(P_ens_test_raw, T_conv)
    P_ens_test_stage1          = mix3(P_test["SWIN"], P_test["EVA"], P_test["COAT"], w_stage1["w_swin"], w_stage1["w_eva"])
    P_ens_test_twostage_scalar = temp_scale(P_ens_test_stage1, T_ens)
    P_ens_test_class_adaptive  = temp_scale(P_ens_test_stage1, T_vec_class)

    test_ens_probs = {
        "Uncalibrated": P_ens_test_raw,
        "Conventional_TS": P_ens_test_conv,
        "Member_Only_TS": P_ens_test_stage1,
        "TwoStage_Scalar_TS": P_ens_test_twostage_scalar,
        "Proposed_ClassAdaptive_TS": P_ens_test_class_adaptive
    }

    # Final test prediction using the method selected on validation.
    P_ens = test_ens_probs[selected_calib_method]
    sel_w = method_meta[selected_calib_method]["w"]

    # --- 2. Pre-specified five-method comparison on the test set ---
    test_ablation_rows = []
    for m_name, P_ens_t in test_ens_probs.items():
        mt = eval_calib_metrics(y_te, P_ens_t, n_class)
        w_used = method_meta[m_name]["w"]
        w_str = f"[{w_used['w_swin']:.2f}, {w_used['w_eva']:.2f}, {w_used['w_coat']:.2f}]"
        test_ablation_rows.append({
            "Study": study_key, "Split": "TEST", "Method": m_name,
            "Weight_Source": method_meta[m_name]["w_src"], "Weights": w_str,
            "Selected_on_VAL": (m_name == selected_calib_method), **mt
        })

    test_ablation_df = pd.DataFrame(test_ablation_rows)
    print("\n--- [INDEPENDENT TEST] 5-WAY PIPELINE-CONSISTENT CALIBRATION COMPARISON ---")
    print("Note: All fusion weights and calibration parameters were selected on validation; this table is descriptive and must not be used for further model selection.")
    print(test_ablation_df[["Method", "Weight_Source", "Weights", "Selected_on_VAL", "ECE", "NLL", "Brier", "Acc", "Balanced_Acc", "MacroF1", "ROC_AUC"]].to_string(index=False))

    # --- 3. Statistical evaluation: lesion-cluster paired bootstrap CIs and image-level McNemar test ---
    test_lesion_ids = get_lesion_clusters(splits["test"]["ids"], meta_df)
    n_unique_lesions = len(pd.unique(test_lesion_ids))
    print(f"\n[STAT INFO] Test images={len(y_te)} | Unique lesion clusters={n_unique_lesions}")

    stat_rows = []
    for ref_m in ["Uncalibrated", "Conventional_TS", "Member_Only_TS"]:
        stat_rows.extend(paired_bootstrap_ci(
            y_te, test_ens_probs, ref_method=ref_m, target_method="TwoStage_Scalar_TS",
            n_class=n_class, lesion_ids=test_lesion_ids, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE
        ))
    for ref_m in ["Uncalibrated", "Conventional_TS", "Member_Only_TS", "TwoStage_Scalar_TS"]:
        stat_rows.extend(paired_bootstrap_ci(
            y_te, test_ens_probs, ref_method=ref_m, target_method="Proposed_ClassAdaptive_TS",
            n_class=n_class, lesion_ids=test_lesion_ids, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE
        ))
    stat_df = pd.DataFrame(stat_rows)
    print("--- [STATISTICAL ANALYSIS: Lesion-Cluster Paired Bootstrap CI (N=1000) and Image-Level McNemar Test] ---")
    print(stat_df[["Comparison", "N_Clusters", "Metric", "Delta_Mean", "Delta_CI95_Low", "Delta_CI95_High", "McNemar_p_exact"]].to_string(index=False))

    # --- 4. Error diversity analysis on the test set ---
    div_df = compute_error_diversity(y_te, P_test)
    print("\n--- [HETEROGENEOUS BACKBONE ERROR DIVERSITY (TEST)] ---")
    print(div_df.to_string(index=False))

    acc   = accuracy_score(y_te, P_ens.argmax(1))
    mf1   = f1_score(y_te, P_ens.argmax(1), average="macro", zero_division=0)
    top3  = top_k_accuracy_score(y_te, P_ens, k=3)
    orc_a, orc_t3 = oracle_acc_top3(y_te, [P_test["SWIN"], P_test["EVA"], P_test["COAT"]])

    print(f"\n[FINAL SELECTED TEST ({selected_calib_method})] Acc={acc:.4f} | MacroF1={mf1:.4f} | Top3={top3:.4f}")
    print(f"[ORACLE] Acc={orc_a:.4f} | Top3={orc_t3:.4f}")

    out_dir = os.path.join(SAVE_DIR, f"RUN_{study_key}"); os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, "test_proba.npy"), P_ens)
    val_ablation_df.to_csv(os.path.join(out_dir, "CALIBRATION_ABLATION_VAL.csv"), index=False)
    test_ablation_df.to_csv(os.path.join(out_dir, "CALIBRATION_ABLATION_TEST.csv"), index=False)
    stat_df.to_csv(os.path.join(out_dir, "STATISTICAL_TESTS.csv"), index=False)
    div_df.to_csv(os.path.join(out_dir, "ERROR_DIVERSITY_TEST.csv"), index=False)

    pd.DataFrame([{
        "Study": study_key, "Selected_Method": selected_calib_method,
        "Weight_Source": method_meta[selected_calib_method]["w_src"],
        "Acc": acc, "MacroF1": mf1, "Top3": top3,
        "w_swin": sel_w["w_swin"], "w_eva": sel_w["w_eva"], "w_coat": sel_w["w_coat"],
        "w_raw_tuple": json.dumps([w_raw["w_swin"], w_raw["w_eva"], w_raw["w_coat"]]),
        "w_stage1_tuple": json.dumps([w_stage1["w_swin"], w_stage1["w_eva"], w_stage1["w_coat"]]),
        "best_C": aux["COAT"]["hp"]["C"], "T_conv": float(T_conv), "T_ens": float(T_ens),
        "lambda_class_T": float(LAMBDA_CLASS_T), "bound_delta_class_T": float(CLASS_T_BOUND_DELTA),
        "T_vec_class": json.dumps([round(float(x), 4) for x in T_vec_class]),
        "Note": "Deterministic per study. Fusion weights and calibration settings are selected on validation; test metrics are descriptive and are not used for further selection."
    }]).to_csv(os.path.join(out_dir, "SUMMARY.csv"), index=False)

    # --- Confusion matrix, classification report, and rescue attribution for the selected Stage-2 model ---
    y_pred = P_ens.argmax(1)
    save_cm_cr(out_dir, y_te, y_pred, class_names, study_key)

    if selected_calib_method == "Proposed_ClassAdaptive_TS":
        rescue_P_dict = P_test
        rescue_T_stage2 = T_vec_class
    elif selected_calib_method == "TwoStage_Scalar_TS":
        rescue_P_dict = P_test
        rescue_T_stage2 = T_ens
    elif selected_calib_method == "Member_Only_TS":
        rescue_P_dict = P_test
        rescue_T_stage2 = None
    elif selected_calib_method == "Conventional_TS":
        rescue_P_dict = P_test_uncal
        rescue_T_stage2 = T_conv
    else:
        rescue_P_dict = P_test_uncal
        rescue_T_stage2 = None

    save_rescue_reports(
        out_dir, splits["test"]["ids"], class_names, y_te, rescue_P_dict,
        (sel_w["w_swin"], sel_w["w_eva"], sel_w["w_coat"]),
        T_stage2=rescue_T_stage2
    )

    return {"Study":study_key, "Selected_Method": selected_calib_method, "Acc":acc, "MacroF1":mf1, "Top3":top3,
            "Weights":[sel_w["w_swin"], sel_w["w_eva"], sel_w["w_coat"]],
            "C":aux["COAT"]["hp"]["C"], "T_conv": float(T_conv), "T_ens": float(T_ens)}

## Entrypoint and pre-specified A0 vs A4 comparison

Loads the configured data and executes only the requested pre-specified study keys. The output order follows the configured study order, not test performance. The final confusion-matrix/report comparison always uses A0 versus A4, not a test-selected “best” metadata model.

In [ ]:
def main(selected=None):
    splits, class_names = load_global_splits(SPLIT_DIR)
    data = {}
    for feat, path in FEATURE_DIRS.items():
        data[feat] = {
            "Xtr": _load_features(path, "train"),
            "Xva": _load_features(path, "val"),
            "Xte": _load_features(path, "test"),
        }
    for feat in data.keys():
        data[feat]["ytr"] = splits["train"]["y"]
        data[feat]["yva"] = splits["val"]["y"]
        data[feat]["yte"] = splits["test"]["y"]

    meta_df = _read_meta(META_CSV)

    sets = {k:set(v["ids"]) for k,v in splits.items()}
    for (a,b) in itertools.combinations(sets.keys(),2):
        print(f"[LEAK] ID overlap {a}-{b}: {len(sets[a].intersection(sets[b]))}")
    mp = {r["_img"]: r["_region_raw"] for _,r in meta_df.iterrows()}
    def dist(ids): return pd.Series([mp.get(i,"missing") for i in ids]).fillna("missing").astype(str).value_counts()
    tabs=[dist(splits[s]["ids"]) for s in ["train","val","test"]]
    idx = sorted(set().union(*[t.index for t in tabs]))
    table = np.vstack([t.reindex(idx, fill_value=0).values for t in tabs])
    chi2,p,_,_ = chi2_contingency(table)
    print(f"[DIST] region chi2={chi2:.1f}, p={p:.3e}")

    run_list_all = list(VALID_STUDIES.keys())
    if selected is None or len(selected) == 0:
        run_list = run_list_all
    else:
        run_list = [s for s in selected if s in run_list_all]

    rows=[]
    for sk in run_list:
        assert sk in VALID_STUDIES, f"Invalid study key: {sk}"
        rows.append(run_one_study(sk, data, class_names, splits, meta_df))

    # Preserve the pre-specified study order; never rank study configurations by test metrics.
    df = pd.DataFrame(rows)
    if not df.empty:
        study_order = {study_key: i for i, study_key in enumerate(run_list)}
        df["_study_order"] = df["Study"].map(study_order)
        df = df.sort_values("_study_order").drop(columns="_study_order")
    df.to_csv(os.path.join(SAVE_DIR, "STUDY_COMPARISON.csv"), index=False)
    print("\n===== PRE-SPECIFIED A0 VS A4 COMPARISON =====")
    print(df.to_string(index=False))
    print(f"\nAll outputs saved to: {SAVE_DIR}")

    try:
        comp = pd.read_csv(os.path.join(SAVE_DIR, "STUDY_COMPARISON.csv"))
        A0_key = "A0_BaseOnly"
        if A0_key not in comp["Study"].values:
            raise FileNotFoundError(f"{A0_key} results were not found.")

        A0_dir = os.path.join(SAVE_DIR, f"RUN_{A0_key}")
        y_true = splits["test"]["y"]
        P_A0 = np.load(os.path.join(A0_dir, "test_proba.npy"))
        y_pred_A0 = P_A0.argmax(1)

        # A4 is a pre-specified comparison model. Do not select the metadata model using test scores.
        A4_key = "A4_CHAMPION_G3_R_full"
        if A4_key not in comp["Study"].values:
            raise FileNotFoundError(f"Pre-specified comparison study {A4_key} was not run.")

        A4_dir = os.path.join(SAVE_DIR, f"RUN_{A4_key}")
        P_A4 = np.load(os.path.join(A4_dir, "test_proba.npy"))
        y_pred_A4 = P_A4.argmax(1)

        write_compare_cm_cr(SAVE_DIR, "A0_vs_A4_PRE_SPECIFIED", class_names, y_true, y_pred_A0, y_true, y_pred_A4,
                            A_name=A0_key, B_name=A4_key)
        print(f"[COMPARE] Confusion matrix and classification report were written for A0 vs {A4_key}.")
    except Exception as e:
        print(f"[COMPARE] Comparison could not be written (perhaps only one study was run): {e}")

if __name__ == "__main__":
    main()